# Oppitunti 03 - Agenttisuunnittelumallit

Tässä oppitunnissa tutustumme kolmeen perustavanlaatuiseen suunnittelumalliin tehokkaiden tekoälyagenttien rakentamiseksi:

1. **Selkeät agenttiohjeet** — Tarkkojen, roolia määrittävien kehotteiden laatiminen agentin käyttäytymisen ohjaamiseksi
2. **Rakenneellinen tuloste Pydantic-mallien avulla** — Varmistetaan, että agentit palauttavat ennustettavissa olevaa, validoitua dataa
3. **Yhden vastuun agentit** — Suunnitellaan keskittyneitä agenteja, jotka tekevät kunkin yhden asian hyvin

Sovellamme kutakin mallia **matkakohdesuositusjärjestelmässä**, rakentamalla vaiheittain järjestelmä, joka voi ehdottaa kohteita, tarkistaa saatavuuden ja hoitaa logistiikan.


## Asennus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kaava 1: Selkeät ohjeet agentille

Vaikutuksiltaan merkittävin kaava on myös yksinkertaisin: kirjoita selkeät, yksityiskohtaiset ohjeet agentillesi.

Hyvät ohjeet määrittelevät:
- **Kuka** agentti on (persoonallisuus ja sävy)
- **Mitä** sen tulee tehdä (vastuulliset tehtävät askel askeleelta)
- **Miten** sen tulee käyttäytyä (rajoitteet ja tyyli)

Alla luomme matkakonsiergi-agentin, jolla on eksplisiittiset ohjeet, jotka muokkaavat jokaista sen tuottamaa vastausta.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Malli 2: Rakenteinen tulostus Pydantic-mallien avulla

Vapaamuotoinen teksti on hyödyllistä keskustelussa, mutta loppujärjestelmät tarvitsevat rakenteellista dataa.
Yhdistämällä **Pydantic-mallit** ja **työkalufunktio** voimme:

- Määritellä tarkka skeema agentin tulostetta varten
- Tarkistaa vastaukset automaattisesti
- Integroida agentin tulokset sovelluslogiikkaan luotettavasti

Toteutuksen avain on `response_format` -parametrin välittäminen agentin suorittamisen yhteydessä. Tämä pakottaa
mallin palauttamaan validoidun `TravelRecommendations`-olion (joka on saatavilla `response.value`)
vapaamuotoisen tekstin sijaan. `get_destination_details` -työkalu palauttaa myös tyypitetyn
`DestinationRecommendation`-olion, joten data pysyy rakenteellisena päästä päähän.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Kuvio 3: Yhden Vastuun Agentit

Monimutkaiset tehtävät hyötyvät työn jakamisesta useille keskittyneille agenteille, joista jokaisella on yksi vastuualue:

- **Kohdeasiantuntija**, joka tuntee paikat ja saatavuuden
- **Logistiikkasuunnittelija**, joka hoitaa lennot, hotellit ja matkasuunnitelmat

Tämä heijastaa ohjelmistokehityksen periaatetta *erillään pitämisestä* — jokaista agenttia on helpompi testata, ylläpitää ja parantaa itsenäisesti.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Yhteenveto

Tässä oppitunnissa sovelsimme kolmea agenttisuunnittelumallia matkasuositusten skenaariossa:

| Malli | Keskeinen ajatus | Hyöty |
|---|---|---|
| **Selkeät ohjeet** | Määrittele persoona, vastuut ja rajoitteet etukäteen | Johdonmukainen ja brändin mukainen agentin käyttäytyminen |
| **Rakenteinen vastaus** | Käytä Pydantic-malleja vastausmuotona | Validioidut, koneellisesti luettavat tulokset |
| **Yksi vastuualue** | Anna jokaiselle agentille yksi keskittynyt tehtävä | Helpompi testata, ylläpitää ja yhdistellä |

Nämä mallit yhdistyvät luonnollisesti — voit yhdistää selkeät ohjeet ja rakenteisen vastauksen yhden vastuualueen agentin sisällä rakentaaksesi vankkoja, tuotantovalmiita järjestelmiä.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
